<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!git clone https://github.com/yaranoun/ML-Tech.git

fatal: destination path 'ML-Tech' already exists and is not an empty directory.


In [30]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 6), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 3.16 KiB | 647.00 KiB/s, done.
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
   fa84563..1f79a10  main       -> origin/main
Updating fa84563..1f79a10
Fast-forward
 data/processed/chunks.json    |  31 ++++-
 notebooks/02_embeddings.ipynb | 263 ++++++++++++++++++++----------------------
 2 files changed, 152 insertions(+), 142 deletions(-)


In [31]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 20 chunks
{'document': 'Personal attendance required.txt', 'title': 'Personal attendance required', 'url': 'https://www.general-security.gov.lb/en/posts/73', 'category': 'Personal attendance required', 'keywords': 'Lebanese citizens, minors, exemption from attendance, exemption from fees', 'section': 'Personal attendance required', 'text': 'Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.\nMinors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent signed the letter at the mayor’s office.\nMin

In [32]:
!pip install -q sentence-transformers faiss-cpu

In [33]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [34]:
texts = [chunk["text"] for chunk in chunks]
passages = ["passage: " + text for text in texts]

In [35]:
embeddings = model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(20, 768)


In [36]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [40]:
faiss.write_index(
    index,
    "data/processed/passport_index.faiss"
)

In [37]:
def retrieve(question, k=3):
    query_embedding = model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"]
        })

    return results

In [39]:
results = retrieve("Is it necessary for me to physically attend to get my passport?")

for result in results:
    print(result["score"])
    print(result["document"])
    print(result["section"])
    print(result["text"])
    print()

0.8466050624847412
Ex-porting Biometric Passport.txt
For the individual planning on shipping his passport with another traveler
1-   The owner needs to show up in person at the department of press – general security, with the traveler concerned, to convey the pre-mentioned request.
2-   The traveler has to have his airplane ticket in hand to underline the date of his departure, as well as proof of an entry visa and a stable residence in the country of destination
3-   The traveler is held responsible in case of losing the passport, on in case of any illegal use of the latter

0.8443633913993835
Personal attendance required.txt
Personal attendance required
Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.
Minors aged 7 years or younger have to accompany their parents to the mayor’s offic